# Estimasi Nilai Pi dengan Monte Carlo Berurutan dan Paralel

## 1. Latar Belakang dan Rumusan Masalah

### Latar Belakang
Metode Monte Carlo memperkirakan nilai pi dengan membangkitkan titik acak di persegi satuan [0,1] x [0,1] dan menghitung proporsi titik yang jatuh di dalam seperempat lingkaran (jarak ke titik asal <= 1). Rasio titik di dalam lingkaran terhadap total titik, dikalikan 4, mendekati nilai pi.

Metode ini cocok untuk demonstrasi paralelisasi karena setiap chunk titik dapat dihitung secara independen (embarrassingly parallel).

### Rumusan Masalah
1. Bagaimana metode Monte Carlo dapat digunakan untuk mengestimasi nilai pi?
2. Apakah pemrosesan paralel dengan dua worker pada Colab CPU menghasilkan percepatan dibanding pemrosesan sekuensial?

## 2. Tujuan dan Batasan Studi Kasus

### Tujuan
1. Mendemonstrasikan estimasi pi dengan Monte Carlo menggunakan NumPy.
2. Membandingkan waktu eksekusi pemrosesan sekuensial dan paralel (2 worker).
3. Menghitung galat terhadap `numpy.pi` dan speedup.
4. Menjelaskan hubungan konsep pembagian kerja dengan HPC.

### Batasan
- Dijalankan pada **Google Colab CPU gratis**, bukan superkomputer.
- Hanya dua ukuran sampel ringan: 200.000 dan 1.000.000 titik.
- Jumlah worker paralel: 2.
- Tidak menjamin bahwa paralel pasti lebih cepat — overhead pembuatan proses bisa mendominasi.
- Seed tetap digunakan agar hasil dapat direproduksi.

## 4. Data dan Asumsi

### Data
Tidak ada dataset eksternal. Data berupa **titik acak** yang dibangkitkan menggunakan NumPy:
- Koordinat (x, y) dengan distribusi uniform di [0, 1].
- Dua ukuran sampel: N = 200.000 dan N = 1.000.000.

### Chunk dan Seed
Untuk perbandingan yang adil, **daftar chunk dan seed yang sama** digunakan pada metode sekuensial dan paralel:
- N = 200.000 dibagi menjadi 2 chunk: [100.000, 100.000] dengan seed [42, 43].
- N = 1.000.000 dibagi menjadi 2 chunk: [500.000, 500.000] dengan seed [42, 43].

### Asumsi
- Hasil Monte Carlo bersifat probabilistik; galat berkurang seiring bertambahnya titik.
- Overhead paralelisasi (pembuatan proses, serialisasi data) dapat membuat paralel lebih lambat pada beban ringan.

## 4. Metode dan Alur Penyelesaian

1. **Buat modul worker** (`mc_worker.py`) yang berisi fungsi untuk menghitung jumlah titik di dalam seperempat lingkaran.
2. **Sekuensial**: Panggil fungsi worker untuk setiap chunk secara berurutan.
3. **Paralel**: Gunakan `ProcessPoolExecutor` dengan 2 worker untuk memproses chunk secara bersamaan.
4. **Pengukuran waktu**: Gunakan `time.perf_counter`, lakukan 3 ulangan setelah 1 warm-up, ambil median.
5. **Evaluasi**: Hitung estimasi pi, galat terhadap `numpy.pi`, dan speedup (waktu sekuensial / waktu paralel).
6. **Visualisasi**: Grafik perbandingan waktu.

## 5. Implementasi Python

### 6.1 Buat Modul Worker (`mc_worker.py`)

Fungsi worker disimpan dalam file `.py` terpisah agar dapat diimpor oleh `ProcessPoolExecutor`. Ini menghindari masalah pickling pada fungsi lambda atau fungsi lokal.

In [ ]:
# Buat file mc_worker.py yang berisi fungsi worker
worker_code = '''
import numpy as np

def estimate_pi_chunk(args):
    # Hitung jumlah titik di dalam seperempat lingkaran.
    # args: tuple (n_points, seed)
    # returns: int, jumlah titik di dalam seperempat lingkaran
    n_points, seed = args
    rng = np.random.RandomState(seed)
    x = rng.uniform(0, 1, size=n_points)
    y = rng.uniform(0, 1, size=n_points)
    inside = np.sum(x**2 + y**2 <= 1.0)
    return inside
'''

with open('mc_worker.py', 'w') as f:
    f.write(worker_code)

print("File 'mc_worker.py' berhasil dibuat.")
print("Isi file:")
print(worker_code)

### 6.2 Import Library dan Konfigurasi

In [ ]:
import numpy as np
import time
import os
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
from mc_worker import estimate_pi_chunk
import matplotlib.pyplot as plt

# Informasi sistem
n_cpu = os.cpu_count()
print(f"Jumlah CPU yang terdeteksi: {n_cpu}")
print(f"Jumlah worker paralel yang akan digunakan: 2")
print(f"numpy.pi = {np.pi}")

### 6.3 Konfigurasi Eksperimen

In [ ]:
# Konfigurasi eksperimen
experiments = [
    {'label': 'N=200.000', 'chunks': [(100_000, 42), (100_000, 43)], 'total': 200_000},
    {'label': 'N=1.000.000', 'chunks': [(500_000, 42), (500_000, 43)], 'total': 1_000_000},
]

N_WORKERS = 2
N_REPEATS = 3  # ulangan pengukuran (setelah warm-up)
N_WARMUP = 1   # warm-up sebelum pengukuran

print("Konfigurasi eksperimen:")
for exp in experiments:
    print(f"  {exp['label']}: {len(exp['chunks'])} chunk, total {exp['total']:,} titik")
print(f"  Worker paralel: {N_WORKERS}")
print(f"  Ulangan: {N_WARMUP} warm-up + {N_REPEATS} pengukuran, ambil median")

### 6.4 Fungsi Pemrosesan Sekuensial dan Paralel

In [ ]:
def run_sequential(chunks):
    # Jalankan estimasi pi secara sekuensial.
    total_inside = 0
    total_points = 0
    for chunk_args in chunks:
        inside = estimate_pi_chunk(chunk_args)
        total_inside += inside
        total_points += chunk_args[0]
    pi_est = 4.0 * total_inside / total_points
    return pi_est

def run_parallel(chunks, n_workers=2):
    # Jalankan estimasi pi secara paralel dengan ProcessPoolExecutor.
    total_points = sum(c[0] for c in chunks)
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        results = list(executor.map(estimate_pi_chunk, chunks))
    total_inside = sum(results)
    pi_est = 4.0 * total_inside / total_points
    return pi_est

def measure_time(func, chunks, n_warmup=1, n_repeats=3, **kwargs):
    # Ukur waktu eksekusi dengan warm-up dan ulangan.
    # Warm-up
    for _ in range(n_warmup):
        func(chunks, **kwargs)

    # Pengukuran
    times = []
    pi_est = None
    for _ in range(n_repeats):
        start = time.perf_counter()
        pi_est = func(chunks, **kwargs)
        end = time.perf_counter()
        times.append(end - start)

    return np.median(times), pi_est

print("Fungsi pemrosesan siap.")

### 6.5 Jalankan Eksperimen

In [ ]:
results = []

for exp in experiments:
    print(f"\n{'='*60}")
    print(f"Eksperimen: {exp['label']}")
    print(f"{'='*60}")

    # Sekuensial
    t_seq, pi_seq = measure_time(run_sequential, exp['chunks'],
                                  n_warmup=N_WARMUP, n_repeats=N_REPEATS)
    err_seq = abs(pi_seq - np.pi)

    # Paralel
    t_par, pi_par = measure_time(run_parallel, exp['chunks'],
                                  n_warmup=N_WARMUP, n_repeats=N_REPEATS,
                                  n_workers=N_WORKERS)
    err_par = abs(pi_par - np.pi)

    speedup = t_seq / t_par if t_par > 0 else float('inf')

    print(f"\n  Sekuensial:")
    print(f"    Pi estimasi : {pi_seq:.8f}")
    print(f"    Galat       : {err_seq:.8f}")
    print(f"    Waktu median: {t_seq:.6f} detik")

    print(f"\n  Paralel ({N_WORKERS} worker):")
    print(f"    Pi estimasi : {pi_par:.8f}")
    print(f"    Galat       : {err_par:.8f}")
    print(f"    Waktu median: {t_par:.6f} detik")

    print(f"\n  Speedup (sekuensial/paralel): {speedup:.4f}x")

    if speedup < 1.0:
        print("  >> Paralel LEBIH LAMBAT — overhead pembuatan proses mendominasi.")
    elif speedup > 1.0:
        print("  >> Paralel lebih cepat.")
    else:
        print("  >> Waktu hampir sama.")

    results.append({
        'label': exp['label'],
        'total': exp['total'],
        't_seq': t_seq, 'pi_seq': pi_seq, 'err_seq': err_seq,
        't_par': t_par, 'pi_par': pi_par, 'err_par': err_par,
        'speedup': speedup
    })

### 6.6 Tabel Hasil

In [ ]:
import pandas as pd

df_results = pd.DataFrame(results)
df_display = df_results[['label', 'total', 'pi_seq', 'err_seq', 't_seq',
                          'pi_par', 'err_par', 't_par', 'speedup']].copy()
df_display.columns = ['Eksperimen', 'Total Titik', 'Pi (Sekuensial)', 'Galat (Seq)',
                       'Waktu Seq (s)', 'Pi (Paralel)', 'Galat (Par)',
                       'Waktu Par (s)', 'Speedup']

print("Tabel Hasil Eksperimen:")
df_display.round(8)

### 6.7 Grafik Perbandingan Waktu

In [ ]:
labels = [r['label'] for r in results]
t_seq_list = [r['t_seq'] for r in results]
t_par_list = [r['t_par'] for r in results]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, t_seq_list, width, label='Sekuensial', color='steelblue')
bars2 = ax.bar(x + width/2, t_par_list, width, label=f'Paralel ({N_WORKERS} worker)', color='coral')

# Label di atas bar
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{bar.get_height():.4f}s', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{bar.get_height():.4f}s', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Waktu Median (detik)')
ax.set_title('Perbandingan Waktu: Sekuensial vs Paralel')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
plt.tight_layout()
plt.savefig('grafik_waktu_monte_carlo.png', dpi=150, bbox_inches='tight')
print("Grafik disimpan sebagai 'grafik_waktu_monte_carlo.png'")
plt.show()

### 6.8 Ekspor Hasil ke CSV

In [ ]:
df_display.to_csv('hasil_monte_carlo.csv', index=False)
print("Tabel hasil disimpan sebagai 'hasil_monte_carlo.csv'")

## 6. Pembahasan

### Menjawab Rumusan Masalah 1: Estimasi Pi dengan Monte Carlo

Metode Monte Carlo berhasil menghasilkan estimasi yang mendekati nilai pi sebenarnya. Galat berkurang seiring bertambahnya jumlah titik, sesuai sifat konvergensi Monte Carlo yang berbanding terbalik dengan akar jumlah sampel.

### Menjawab Rumusan Masalah 2: Apakah Paralel Lebih Cepat?

Pada lingkungan Google Colab CPU gratis, hasil mungkin menunjukkan salah satu dari:
- **Paralel lebih lambat** (speedup < 1) — Ini terjadi karena overhead pembuatan proses (`ProcessPoolExecutor` membuat proses baru, melakukan serialisasi fungsi dan data) mendominasi waktu komputasi pada beban yang relatif ringan.
- **Paralel sedikit lebih cepat** (speedup > 1) — Bisa terjadi pada ukuran sampel yang lebih besar jika waktu komputasi sudah cukup mendominasi overhead.

**Jangan menjamin percepatan.** Pada HPC sesungguhnya dengan jutaan sampel dan ratusan node, keuntungan paralelisasi jauh lebih terasa. Pada Colab dengan 2 worker dan beban ringan, overhead bisa mendominasi.

### Hubungan dengan Materi HPC

Pemateri menjelaskan bahwa HPC membagi pekerjaan ke banyak node yang bekerja bersamaan melalui jaringan berkecepatan tinggi. Demonstrasi ini menunjukkan konsep yang sama pada skala kecil:
- Setiap chunk adalah "pekerjaan" yang bisa dikirim ke node berbeda.
- `ProcessPoolExecutor` berperan sebagai pengelola distribusi pekerjaan.
- Hasil dari setiap chunk digabungkan untuk mendapatkan jawaban akhir.

Perbedaan utama: HPC memiliki infrastruktur jaringan cepat dan proses yang sudah berjalan di masing-masing node, sehingga overhead distribusi jauh lebih kecil dibanding `multiprocessing` di satu mesin.

## 7. Keterbatasan dan Kesimpulan

### Keterbatasan
1. **Lingkungan Colab CPU** — Bukan HPC. Overhead multiprocessing signifikan pada beban ringan.
2. **Hanya 2 worker** — Tidak merepresentasikan paralelisasi berskala besar.
3. **Beban ringan** — 200K dan 1M titik relatif kecil; pada HPC nyata, beban bisa miliaran.
4. **Variabilitas waktu** — Waktu eksekusi di Colab bisa bervariasi tergantung beban server Google.
5. **Speedup tidak dijamin** — Tergantung rasio waktu komputasi vs overhead.

### Kesimpulan
1. Metode Monte Carlo berhasil mengestimasi pi dengan galat yang berkurang seiring penambahan titik.
2. Konsep pembagian pekerjaan (chunking) berhasil didemonstrasikan pada sekuensial dan paralel.
3. Pada Colab CPU gratis, overhead paralelisasi mungkin lebih besar dari manfaatnya untuk beban ringan.
4. Pada skala HPC sesungguhnya, manfaat paralelisasi jauh lebih signifikan.
5. Sesuai pesan pemateri: mahasiswa perlu memahami konsep pembagian kerja dan infrastruktur komputasi, tidak hanya menjalankan kode.